# 1) About

## Purpose

This notebook establishes a **rule-based baseline** for AML fraud detection before building ML models. Rules provide a simple, interpretable floor that ML models must beat to justify their complexity.

## Connection to Feature Engineering
We will use the dataset we have produced in feature engineering notebook. 

- 29,997 rows (9999 accounts x 3 monthly snapshots)
- 28 behavioral features
- ~2.9% positive label rate

## Approach

1. Time based train/test split (Train: Apr, May. Test: June.)
2. Define evaluation framework (Precision@TopK, Recall@TopK)
3. Design simple rules based on EDA findings (fan-in)
4. Apply to test set and establish the baseline to beat

## Why rules first?

- **Interpretable:** Investigators can understand exactly why each account was flagged
- **Baseline for ML** If ML doesn't beat rules, the complexity is not justified
- **Production reality** Most banks start with rules and add ML on top.

## Why time-based split for rules?

Rule based models don't learn like ML but there are key reasons why I am moving forward with time based split:

1. Threshold Selection: Even though rules don't train, we pick thresholds looking at training data distributions. If I choose thresholds using the test set, result would artifically look good. Kind of data leakage.
2. Fair Comparison: Since we are going to compare ML vs. Rule Based approach later, both must be evaluated on the same held-out dataset.

-> Pick thresolds using training data, evaluate only on test data.


# 2) Imports and Data Loading

In [19]:

import pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load the data
data_path = Path("../data/processed/modeling_dataset.csv")
df = pd.read_csv(data_path)
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])

# Inspect

print(f"Shape: {df.shape}")
print(f"\nSnapshots: {df['snapshot_date'].unique()}")
print(f"\nLabel Disribution: \n{df['label'].value_counts()}")
print(f"\nPositiveRate: {df['label'].mean()*100:.2f}")



Shape: (29997, 31)

Snapshots: <DatetimeArray>
['2020-04-01 00:00:00', '2020-05-01 00:00:00', '2020-06-01 00:00:00']
Length: 3, dtype: datetime64[us]

Label Disribution: 
label
0    29128
1      869
Name: count, dtype: int64

PositiveRate: 2.90


# 3) Train-Test Split (Time Based)

We will split data by snapshot date.

-> Train: April + May Snapshots - to pick the thresholds

-> Test: June snapshot - for final evaluation

This split stimulates production.

In [20]:
train_df = df[df['snapshot_date'] < '2020-06-01'].copy()
test_df = df[df['snapshot_date'] >= '2020-06-01'].copy()

print(f"Train: {len(train_df):,} rows | {train_df['label'].sum()} fraudulent clients | {train_df['label'].mean()*100:.2f}% fraudulent rate")
print(f"Test:  {len(test_df):,} rows | {test_df['label'].sum()} fraud ({test_df['label'].mean()*100:.2f}%)")


Train: 19,998 rows | 527 fraudulent clients | 2.64% fraudulent rate
Test:  9,999 rows | 342 fraud (3.42%)


# 4) Evaluation Framework

Let's define the evaluation frameworks before writing the rules.

Rules produce **binary flags** — each account is either alerted or not. So the right questions are: of the accounts we flag, how many are true positives? And of all the positive cases, how many do we catch?

**Metrics:**

- `precision` from the accounts we alert, what % are true positives? (Workload quality)
- `recall` of all Positive cases, what % do we catch? (coverage)

In [64]:

def evaluate_rule(flags, labels, rule_name = 'rule'):
    """
    Evaluate a binary rule flag against ground truth labels we have created in feature engineering nb (`label`)
    Params
    ------
    flags    : array-like binary rule output (1: alert, 0: not)
    labels   : array-like ground truth binary labels (1: Positive)
    rule_name: str
    """

    flags = np.array(flags)
    labels = np.array(labels)

    tp = int(((flags==1) & (labels == 1)).sum())
    fp = int(((flags==1) & (labels ==0)).sum())
    fn = int(((flags == 0) & (labels == 1)).sum())
    number_alerts = tp+fp
    total_positives = tp + fn

    precision = tp / number_alerts if number_alerts > 0 else 0.0
    recall = tp / total_positives if number_alerts > 0 else 0.0

    print(f"\n{rule_name}")
    # print(f"\n Alerts created: {number_alerts:,}, {number_alerts/len(flags)*100:.1f}% of all accounts")
    print(f"Precision: {precision:.2%}")
    print(f"Recall: {recall:.2%} ")
    print(f" TP = {tp} ||| FP = {fp} ||| FN = {fn}")
    
    return dict(rule=rule_name, number_alerts=number_alerts, precision=round(precision, 2), recall = round(recall, 2),
                tp=tp, fp=fp, fn=fn)


my_flags = [0, 1, 0, 1]
my_labels = [1,1,1,1]

evaluate_rule(my_flags, my_labels, 'thisisarule')



thisisarule
Precision: 100.00%
Recall: 50.00% 
 TP = 2 ||| FP = 0 ||| FN = 2


{'rule': 'thisisarule',
 'number_alerts': 2,
 'precision': 1.0,
 'recall': 0.5,
 'tp': 2,
 'fp': 0,
 'fn': 2}

In [68]:

def evaluate_rule(flags, labels, rule_name = 'rule'):
    """
    Evaluate a binary rule flag against ground truth labels we have created in feature engineering nb (`label`)
    Params
    ------
    flags    : array-like binary rule output (1: alert, 0: not)
    labels   : array-like ground truth binary labels (1: Positive)
    rule_name: str
    """

    flags = np.array(flags)
    labels = np.array(labels)

    tp = int(((flags==1) & (labels == 1)).sum())
    fp = int(((flags==1) & (labels ==0)).sum())
    fn = int(((flags == 0) & (labels == 1)).sum())
    number_alerts = tp+fp
    total_positives = tp + fn

    precision = tp / number_alerts if number_alerts > 0 else 0.0
    recall = tp / total_positives if number_alerts > 0 else 0.0

    print(f"\n{rule_name}")
    # print(f"\n Alerts created: {number_alerts:,}, {number_alerts/len(flags)*100:.1f}% of all accounts")
    print(f"Precision: {precision:.2%}")
    print(f"Recall: {recall:.2%} ")
    print(f" TP = {tp} ||| FP = {fp} ||| FN = {fn}")
    
    return pd.DataFrame([dict(rule=rule_name, number_alerts=number_alerts, precision=round(precision, 2), recall = round(recall, 2),
                tp=tp, fp=fp, fn=fn)])


my_flags = [0, 1, 0, 1]
my_labels = [1,1,1,1]

evaluate_rule(my_flags, my_labels, 'thisisarule')



thisisarule
Precision: 100.00%
Recall: 50.00% 
 TP = 2 ||| FP = 0 ||| FN = 2


,rule,number_alerts,precision,recall,tp,fp,fn
0,thisisarule,2,1.00,0.50,2,0,2
